Code to add to the raw dlc h5 data a new 'bodypart' named weigthed, which is the nose bodypart if its likelihood is greater than 0.3 and a weigted sum according to the likelihoods of all other bodyparts if it less than 0.3.
We then can define this bodypart in the NaviGraph to get less NaN values in the result (now we'll get Nan only for row where all values in DC where NaN)

In [22]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt


In [31]:
folder_path = "/var/home/almogmeir/Documents/M.Sc/Project/DLC/12NPM_03_12_25_2025_03_12_10_58_38/"
file_name = "12NPM_03_12_25_2025_03_12_10_58_38_cam1DLC_Resnet101_NeuroMazeMar26shuffle7_snapshot_170"

In [32]:
# Read the h5 file, update the path to the raw h5 DLC outputs
h5_file = pd.read_hdf(folder_path + file_name + ".h5")

In [33]:
h5_file

scorer    DLC_Resnet101_NeuroMazeMar26shuffle7_snapshot_170              \
bodyparts                                              nose               
coords                                                    x           y   
0                                               1297.983154  983.381165   
1                                               1297.806885  983.313232   
2                                               1297.884521  983.149963   
3                                               1297.929077  983.143921   
4                                               1298.071167  983.272278   
...                                                     ...         ...   
143548                                          1298.862549  985.579956   
143549                                           410.521240  496.422302   
143550                                          1298.880005  985.566162   
143551                                          1298.762939  985.504517   
143552                                          1298.861206  985.582581   

scorer                                                                \
bodyparts                   ear_L                              ear_R   
coords    likelihood            x           y likelihood           x   
0           0.431004  1301.035278  989.675110   0.289643   21.653118   
1           0.439588  1300.918945  989.591187   0.289184   21.494226   
2           0.440869  1300.909668  989.560181   0.294290   21.547333   
3           0.439708  1300.972046  989.524048   0.292058   21.547070   
4           0.433305  1301.043213  989.605774   0.288232   21.660812   
...              ...          ...         ...        ...         ...   
143548      0.424271   386.129974  504.064331   0.893327  401.267181   
143549      0.458756   386.586365  503.284851   0.938769  402.240509   
143550      0.425120   387.366455  502.538696   0.905918  403.342651   
143551      0.430501   388.013794  501.874146   0.883200  403.840210   
143552      0.424186   389.599060  501.076813   0.891525  404.555450   

scorer                                                                  \
bodyparts                             mid_body                           
coords               y likelihood            x            y likelihood   
0          1006.018494   0.296977  1226.836914    32.060989   0.244646   
1          1005.729980   0.284459     4.938797  1018.155212   0.229472   
2          1005.828674   0.283248     4.950881  1018.205200   0.225801   
3          1005.827820   0.282394     4.946735  1018.220093   0.224703   
4          1006.043152   0.292009     4.816325  1018.304016   0.214242   
...                ...        ...          ...          ...        ...   
143548      519.348633   0.942303   378.786194   528.253723   0.772520   
143549      520.156433   0.969324   378.646454   527.830261   0.743751   
143550      518.341431   0.981317   378.386292   527.030457   0.709996   
143551      518.144714   0.982116   378.901367   526.340454   0.719511   
143552      517.431152   0.972554   379.564880   524.288330   0.758008   

scorer                                                                 \
bodyparts    tail_base                         head_stage               
coords               x           y likelihood           x           y   
0          1212.028687   28.951492   0.418888    9.371573  982.402710   
1          1211.572998   28.360371   0.425376    9.244394  982.424316   
2          1211.583740   28.429068   0.426016    9.235188  982.434509   
3          1211.598633   28.423143   0.423962    9.234289  982.404175   
4          1211.569336   28.364849   0.421606    9.320097  982.334961   
...                ...         ...        ...         ...         ...   
143548      358.325134  550.428040   0.571847  414.531860  496.007355   
143549      358.882996  549.892883   0.562917  414.725494  496.876526   
143550      358.746521  548.639221   0.626313  416.623474  495.577667   
143551      359.702393  548

In [34]:
# Update to be the actual file name according to the first index column value of the h5 table
file_name = 'DLC_Resnet101_NeuroMazeMar26shuffle7_snapshot_170'

# Load the DataFrame from the HDF5 file
dlc_raw_h5 = h5_file[file_name]

# Extract the MultiIndex from the first two rows
multi_index = pd.MultiIndex.from_arrays([dlc_raw_h5.iloc[0], dlc_raw_h5.iloc[1]])

# Convert the data to numeric, errors='coerce' will turn non-numeric values into NaN
dlc_raw_h5 = dlc_raw_h5.apply(pd.to_numeric, errors='coerce')

new_index = pd.MultiIndex.from_tuples([
        ('weighted', 'x'),
        ('weighted', 'y'),
        ('weighted', 'likelihood')
    ])

new_index_df = pd.DataFrame(np.nan, index=range(len(dlc_raw_h5)), columns=new_index)


# Concatenate the original DataFrame with the new columns DataFrame
# dlc_raw_h5 = pd.concat([dlc_raw_h5, new_columns], axis=1)

dlc_raw_h5 = pd.concat([dlc_raw_h5, new_index_df], axis=1)
dlc_raw_h5

nose                               ear_L              \
                  x           y likelihood            x           y   
0       1297.983154  983.381165   0.431004  1301.035278  989.675110   
1       1297.806885  983.313232   0.439588  1300.918945  989.591187   
2       1297.884521  983.149963   0.440869  1300.909668  989.560181   
3       1297.929077  983.143921   0.439708  1300.972046  989.524048   
4       1298.071167  983.272278   0.433305  1301.043213  989.605774   
...             ...         ...        ...          ...         ...   
143548  1298.862549  985.579956   0.424271   386.129974  504.064331   
143549   410.521240  496.422302   0.458756   386.586365  503.284851   
143550  1298.880005  985.566162   0.425120   387.366455  502.538696   
143551  1298.762939  985.504517   0.430501   388.013794  501.874146   
143552  1298.861206  985.582581   0.424186   389.599060  501.076813   

                        ear_R                             mid_body  ...  \
       likelihood           x            y likelihood            x  ...   
0        0.289643   21.653118  1006.018494   0.296977  1226.836914  ...   
1        0.289184   21.494226  1005.729980   0.284459     4.938797  ...   
2        0.294290   21.547333  1005.828674   0.283248     4.950881  ...   
3        0.292058   21.547070  1005.827820   0.282394     4.946735  ...   
4        0.288232   21.660812  1006.043152   0.292009     4.816325  ...   
...           ...         ...          ...        ...          ...  ...   
143548   0.893327  401.267181   519.348633   0.942303   378.786194  ...   
143549   0.938769  402.240509   520.156433   0.969324   378.646454  ...   
143550   0.905918  403.342651   518.341431   0.981317   378.386292  ...   
143551   0.883200  403.840210   518.144714   0.982116   378.901367  ...   
143552   0.891525  404.555450   517.431152   0.972554   379.564880  ...   

                     tail_base                         head_stage              \
       likelihood            x           y likelihood           x           y   
0        0.244646  1212.028687   28.951492   0.418888    9.371573  982.402710   
1        0.229472  1211.572998   28.360371   0.425376    9.244394  982.424316   
2        0.225801  1211.583740   28.429068   0.426016    9.235188  982.434509   
3        0.224703  1211.598633   28.423143   0.423962    9.234289  982.404175   
4        0.214242  1211.569336   28.364849   0.421606    9.320097  982.334961   
...           ...          ...         ...        ...         ...         ...   
143548   0.772520   358.325134  550.428040   0.571847  414.531860  496.007355   
143549   0.743751   358.882996  549.892883   0.562917  414.725494  496.876526   
143550   0.709996   358.746521  548.639221   0.626313  416.623474  495.577667   
143551   0.719511   359.702393  548.057007   0.667162  417.021088  495.295380   
143552   0.758008   359.798431  546.451050   0.638189  417.027588  495.686829   

                  weighted                 
       likelihood        x   y likelihood  
0        0.681662      NaN NaN        NaN  
1        0.679922      NaN NaN        NaN  
2        0.674294      NaN NaN        NaN  
3        0.672150      NaN NaN        NaN  
4        0.674633      NaN NaN        NaN  
...           ...      ...  ..        ...  
143548   0.848309      NaN NaN        NaN  
143549   0.887016      NaN NaN        NaN  
143550   0.922607      NaN NaN        NaN  
143551   0.883615      NaN NaN        NaN  
143552   0.899303      NaN NaN        NaN  

[143553 rows x 21 columns]

In [ ]:
# def calculate_weighted_coordinates(row):
#     body_parts = ['ear_R', 'ear_L', 'mid_body', 'tail_base', 'head_stage']
#     total_weight = 0
#     weighted_x = 0
#     weighted_y = 0
#     weighted_likelihood = 0
#     likelihoods = []

#     for part in body_parts:
#         likelihood = row[part]['likelihood']
#         likelihoods.append(likelihood)
#         if likelihood > 0:
#             weighted_x += row[part]['x'] * likelihood
#             weighted_y += row[part]['y'] * likelihood
#             total_weight += likelihood

#     if total_weight > 0:
#         weighted_x /= total_weight
#         weighted_y /= total_weight
#         weighted_likelihood = sum([likelihood * (likelihood / total_weight) for likelihood in likelihoods])
#     likelihoods = []

#     return (weighted_x, weighted_y, weighted_likelihood)

# # # Apply the function to each row in the dataframe
# # dlc_raw_h5['weighted'][['x', 'y', 'likelihood']] = dlc_raw_h5.apply(
# #     lambda row: calculate_weighted_coordinates(row) if row['nose']['likelihood'] < 0.3 else pd.Series([row['nose']['x'], row['nose']['y'], row['nose']['likelihood']]),
# #     axis=1
# # )

# for i, row in enumerate(dlc_raw_h5.iterrows()):
#     if row[1]['nose']['likelihood'] >= 0.3:
#         dlc_raw_h5.loc[i, ('weighted', ['x', 'y', 'likelihood'])] = (row[1]['nose']['x'], row[1]['nose']['y'], row[1]['nose']['likelihood'])
#     else:
#         dlc_raw_h5.loc[i, ('weighted', ['x', 'y', 'likelihood'])] = calculate_weighted_coordinates(row[1])

# # Display the updated dataframe
# dlc_raw_h5


nose                               ear_L              \
                  x           y likelihood            x           y   
0       1297.983154  983.381165   0.431004  1301.035278  989.675110   
1       1297.806885  983.313232   0.439588  1300.918945  989.591187   
2       1297.884521  983.149963   0.440869  1300.909668  989.560181   
3       1297.929077  983.143921   0.439708  1300.972046  989.524048   
4       1298.071167  983.272278   0.433305  1301.043213  989.605774   
...             ...         ...        ...          ...         ...   
143548  1298.862549  985.579956   0.424271   386.129974  504.064331   
143549   410.521240  496.422302   0.458756   386.586365  503.284851   
143550  1298.880005  985.566162   0.425120   387.366455  502.538696   
143551  1298.762939  985.504517   0.430501   388.013794  501.874146   
143552  1298.861206  985.582581   0.424186   389.599060  501.076813   

                        ear_R                             mid_body  ...  \
       likelihood           x            y likelihood            x  ...   
0        0.289643   21.653118  1006.018494   0.296977  1226.836914  ...   
1        0.289184   21.494226  1005.729980   0.284459     4.938797  ...   
2        0.294290   21.547333  1005.828674   0.283248     4.950881  ...   
3        0.292058   21.547070  1005.827820   0.282394     4.946735  ...   
4        0.288232   21.660812  1006.043152   0.292009     4.816325  ...   
...           ...         ...          ...        ...          ...  ...   
143548   0.893327  401.267181   519.348633   0.942303   378.786194  ...   
143549   0.938769  402.240509   520.156433   0.969324   378.646454  ...   
143550   0.905918  403.342651   518.341431   0.981317   378.386292  ...   
143551   0.883200  403.840210   518.144714   0.982116   378.901367  ...   
143552   0.891525  404.555450   517.431152   0.972554   379.564880  ...   

                     tail_base                         head_stage              \
       likelihood            x           y likelihood           x           y   
0        0.244646  1212.028687   28.951492   0.418888    9.371573  982.402710   
1        0.229472  1211.572998   28.360371   0.425376    9.244394  982.424316   
2        0.225801  1211.583740   28.429068   0.426016    9.235188  982.434509   
3        0.224703  1211.598633   28.423143   0.423962    9.234289  982.404175   
4        0.214242  1211.569336   28.364849   0.421606    9.320097  982.334961   
...           ...          ...         ...        ...         ...         ...   
143548   0.772520   358.325134  550.428040   0.571847  414.531860  496.007355   
143549   0.743751   358.882996  549.892883   0.562917  414.725494  496.876526   
143550   0.709996   358.746521  548.639221   0.626313  416.623474  495.577667   
143551   0.719511   359.702393  548.057007   0.667162  417.021088  495.295380   
143552   0.758008   359.798431  546.451050   0.638189  417.027588  495.686829   

                      weighted                         
       likelihood            x           y likelihood  
0        0.681662  1297.983154  983.381165   0.431004  
1        0.679922  1297.806885  983.313232   0.439588  
2        0.674294  1297.884521  983.149963   0.440869  
3        0.672150  1297.929077  983.143921   0.439708  
4        0.674633  1298.071167  983.272278   0.433305  
...           ...          ...         ...        ...  
143548   0.848309  1298.862549  985.579956   0.424271  
143549   0.887016   410.521240  496.422302   0.458756  
143550   0.922607  1298.880005  985.566162   0.425120  
143551   0.883615  1298.762939  985.504517   0.430501  
143552   0.899303  1298.861206  985.582581   0.424186  

[143553 rows x 21 columns]

In [38]:
def set_likelihood_to_zero_outside_boundary(df, bodypart, x_min, x_max, y_min, y_max):
    """
    Sets the likelihood of a bodypart to 0 if its coordinates are outside the given boundary.

    Parameters:
    df (pd.DataFrame): The dataframe containing the bodypart data.
    bodypart (str): The bodypart to check.
    x_min (float): Minimum x-coordinate boundary.
    x_max (float): Maximum x-coordinate boundary.
    y_min (float): Minimum y-coordinate boundary.
    y_max (float): Maximum y-coordinate boundary.
    """
    outside_boundary = (df[(bodypart, 'x')] < x_min) | (df[(bodypart, 'x')] > x_max) | \
                       (df[(bodypart, 'y')] < y_min) | (df[(bodypart, 'y')] > y_max)
    df.loc[outside_boundary, (bodypart, 'likelihood')] = 0

In [39]:
for bodypart in ['ear_R', 'ear_L', 'mid_body', 'tail_base', 'nose', 'head_stage']:
    set_likelihood_to_zero_outside_boundary(dlc_raw_h5, bodypart, 90, 1170, 70, 930)

In [40]:
def calculate_weighted_coordinates(row):
    nose_likelihood = row['nose']['likelihood']
    head_stage_likelihood = row['head_stage']['likelihood']
    
    # If either nose or head_stage has a likelihood at least 0.3, pick the one with the higher likelihood.
    if nose_likelihood >= 0.3 or head_stage_likelihood >= 0.3:
        if nose_likelihood >= head_stage_likelihood:
            return (row['nose']['x'], row['nose']['y'], nose_likelihood)
        else:
            return (row['head_stage']['x'], row['head_stage']['y'], head_stage_likelihood)
    
    # Both nose and head_stage have low likelihood, so use weighted sum of all body parts.
    body_parts = ['ear_R', 'ear_L', 'mid_body', 'tail_base', 'head_stage']
    total_weight = 0
    weighted_x = 0
    weighted_y = 0
    likelihoods = []

    for part in body_parts:
        likelihood = row[part]['likelihood']
        likelihoods.append(likelihood)
        if likelihood > 0:
            weighted_x += row[part]['x'] * likelihood
            weighted_y += row[part]['y'] * likelihood
            total_weight += likelihood

    if total_weight > 0:
        weighted_x /= total_weight
        weighted_y /= total_weight
        weighted_likelihood = sum(likelihood * (likelihood / total_weight) for likelihood in likelihoods)
    else:
        weighted_likelihood = 0

    return (weighted_x, weighted_y, weighted_likelihood)

for i, row in enumerate(dlc_raw_h5.iterrows()):
    dlc_raw_h5.loc[i, ('weighted', ['x', 'y', 'likelihood'])] = calculate_weighted_coordinates(row[1])

# Display the updated dataframe
dlc_raw_h5

nose                               ear_L              \
                  x           y likelihood            x           y   
0       1297.983154  983.381165   0.000000  1301.035278  989.675110   
1       1297.806885  983.313232   0.000000  1300.918945  989.591187   
2       1297.884521  983.149963   0.000000  1300.909668  989.560181   
3       1297.929077  983.143921   0.000000  1300.972046  989.524048   
4       1298.071167  983.272278   0.000000  1301.043213  989.605774   
...             ...         ...        ...          ...         ...   
143548  1298.862549  985.579956   0.000000   386.129974  504.064331   
143549   410.521240  496.422302   0.458756   386.586365  503.284851   
143550  1298.880005  985.566162   0.000000   387.366455  502.538696   
143551  1298.762939  985.504517   0.000000   388.013794  501.874146   
143552  1298.861206  985.582581   0.000000   389.599060  501.076813   

                        ear_R                             mid_body  ...  \
       likelihood           x            y likelihood            x  ...   
0        0.000000   21.653118  1006.018494   0.000000  1226.836914  ...   
1        0.000000   21.494226  1005.729980   0.000000     4.938797  ...   
2        0.000000   21.547333  1005.828674   0.000000     4.950881  ...   
3        0.000000   21.547070  1005.827820   0.000000     4.946735  ...   
4        0.000000   21.660812  1006.043152   0.000000     4.816325  ...   
...           ...         ...          ...        ...          ...  ...   
143548   0.893327  401.267181   519.348633   0.942303   378.786194  ...   
143549   0.938769  402.240509   520.156433   0.969324   378.646454  ...   
143550   0.905918  403.342651   518.341431   0.981317   378.386292  ...   
143551   0.883200  403.840210   518.144714   0.982116   378.901367  ...   
143552   0.891525  404.555450   517.431152   0.972554   379.564880  ...   

                     tail_base                         head_stage              \
       likelihood            x           y likelihood           x           y   
0        0.000000  1212.028687   28.951492   0.000000    9.371573  982.402710   
1        0.000000  1211.572998   28.360371   0.000000    9.244394  982.424316   
2        0.000000  1211.583740   28.429068   0.000000    9.235188  982.434509   
3        0.000000  1211.598633   28.423143   0.000000    9.234289  982.404175   
4        0.000000  1211.569336   28.364849   0.000000    9.320097  982.334961   
...           ...          ...         ...        ...         ...         ...   
143548   0.772520   358.325134  550.428040   0.571847  414.531860  496.007355   
143549   0.743751   358.882996  549.892883   0.562917  414.725494  496.876526   
143550   0.709996   358.746521  548.639221   0.626313  416.623474  495.577667   
143551   0.719511   359.702393  548.057007   0.667162  417.021088  495.295380   
143552   0.758008   359.798431  546.451050   0.638189  417.027588  495.686829   

                     weighted                         
       likelihood           x           y likelihood  
0        0.000000    0.000000    0.000000   0.000000  
1        0.000000    0.000000    0.000000   0.000000  
2        0.000000    0.000000    0.000000   0.000000  
3        0.000000    0.000000    0.000000   0.000000  
4        0.000000    0.000000    0.000000   0.000000  
...           ...         ...         ...        ...  
143548   0.848309  414.531860  496.007355   0.848309  
143549   0.887016  414.725494  496.876526   0.887016  
143550   0.922607  416.623474  495.577667   0.922607  
143551   0.883615  417.021088  495.295380   0.883615  
143552   0.899303  417.027588  495.686829   0.899303  

[143553 rows x 21 columns]

In [44]:
dlc_raw_h5[dlc_raw_h5['weighted']['likelihood'] < 0.3]

nose                               ear_L              \
                  x           y likelihood            x           y   
0       1297.983154  983.381165        0.0  1301.035278  989.675110   
1       1297.806885  983.313232        0.0  1300.918945  989.591187   
2       1297.884521  983.149963        0.0  1300.909668  989.560181   
3       1297.929077  983.143921        0.0  1300.972046  989.524048   
4       1298.071167  983.272278        0.0  1301.043213  989.605774   
...             ...         ...        ...          ...         ...   
134321  1298.911743  983.838745        0.0  1301.271851  989.713806   
134322  1298.914429  983.822632        0.0  1301.268555  989.731934   
134323  1298.927246  983.787781        0.0  1301.270752  989.691162   
134324  1298.923706  983.770325        0.0  1301.281738  989.694519   
134325  1298.931519  983.774719        0.0  1301.285034  989.627197   

                       ear_R                             mid_body  ...  \
       likelihood          x            y likelihood            x  ...   
0             0.0  21.653118  1006.018494        0.0  1226.836914  ...   
1             0.0  21.494226  1005.729980        0.0     4.938797  ...   
2             0.0  21.547333  1005.828674        0.0     4.950881  ...   
3             0.0  21.547070  1005.827820        0.0     4.946735  ...   
4             0.0  21.660812  1006.043152        0.0     4.816325  ...   
...           ...        ...          ...        ...          ...  ...   
134321        0.0  20.001196  1008.375916        0.0     5.342560  ...   
134322        0.0  20.051632  1008.398254        0.0     5.347792  ...   
134323        0.0  20.039583  1008.390076        0.0     5.346409  ...   
134324        0.0  20.049126  1008.391602        0.0     5.347863  ...   
134325        0.0  20.035990  1008.395081        0.0     5.334172  ...   

                     tail_base                          head_stage  \
       likelihood            x           y likelihood            x   
0             0.0  1212.028687   28.951492        0.0     9.371573   
1             0.0  1211.572998   28.360371        0.0     9.244394   
2             0.0  1211.583740   28.429068        0.0     9.235188   
3             0.0  1211.598633   28.423143        0.0     9.234289   
4             0.0  1211.569336   28.364849        0.0     9.320097   
...           ...          ...         ...        ...          ...   
134321        0.0  1303.627075  978.871216        0.0  1295.810669   
134322        0.0  1303.631958  978.894653        0.0  1295.793823   
134323        0.0  1303.619263  978.880493        0.0  1295.813110   
134324        0.0  1303.628784  978.866394        0.0  1295.812500   
134325        0.0  1303.586670  978.795471        0.0  1295.850464   

                              weighted                  
                 y likelihood        x    y likelihood  
0       982.402710        0.0      0.0  0.0        0.0  
1       982.424316        0.0      0.0  0.0        0.0  
2       982.434509        0.0      0.0  0.0        0.0  
3       982.404175        0.0      0.0  0.0        0.0  
4       982.334961        0.0      0.0  0.0        0.0  
...            ...        ...      ...  ...        ...  
134321  978.494385        0.0      0.0  0.0        0.0  
134322  978.538818        0.0      0.0  0.0        0.0  
134323  978.502747        0.0      0.0  0.0        0.0  
134324  978.504272        0.0      0.0  0.0        0.0  
134325  978.444641        0.0      0.0  0.0        0.0  

[658 rows x 21 columns]

In [45]:
# Restore the top level index (scorer) with the file name
h5_file = pd.concat({file_name: dlc_raw_h5}, axis=1)
# Restore the index column names
h5_file.columns.names = ['scorer', 'bodyparts', 'coords']
h5_file

scorer    DLC_Resnet101_NeuroMazeMar26shuffle7_snapshot_170              \
bodyparts                                              nose               
coords                                                    x           y   
0                                               1297.983154  983.381165   
1                                               1297.806885  983.313232   
2                                               1297.884521  983.149963   
3                                               1297.929077  983.143921   
4                                               1298.071167  983.272278   
...                                                     ...         ...   
143548                                          1298.862549  985.579956   
143549                                           410.521240  496.422302   
143550                                          1298.880005  985.566162   
143551                                          1298.762939  985.504517   
143552                                          1298.861206  985.582581   

scorer                                                                \
bodyparts                   ear_L                              ear_R   
coords    likelihood            x           y likelihood           x   
0           0.000000  1301.035278  989.675110   0.000000   21.653118   
1           0.000000  1300.918945  989.591187   0.000000   21.494226   
2           0.000000  1300.909668  989.560181   0.000000   21.547333   
3           0.000000  1300.972046  989.524048   0.000000   21.547070   
4           0.000000  1301.043213  989.605774   0.000000   21.660812   
...              ...          ...         ...        ...         ...   
143548      0.000000   386.129974  504.064331   0.893327  401.267181   
143549      0.458756   386.586365  503.284851   0.938769  402.240509   
143550      0.000000   387.366455  502.538696   0.905918  403.342651   
143551      0.000000   388.013794  501.874146   0.883200  403.840210   
143552      0.000000   389.599060  501.076813   0.891525  404.555450   

scorer                                          ...                          \
bodyparts                             mid_body  ...               tail_base   
coords               y likelihood            x  ... likelihood            x   
0          1006.018494   0.000000  1226.836914  ...   0.000000  1212.028687   
1          1005.729980   0.000000     4.938797  ...   0.000000  1211.572998   
2          1005.828674   0.000000     4.950881  ...   0.000000  1211.583740   
3          1005.827820   0.000000     4.946735  ...   0.000000  1211.598633   
4          1006.043152   0.000000     4.816325  ...   0.000000  1211.569336   
...                ...        ...          ...  ...        ...          ...   
143548      519.348633   0.942303   378.786194  ...   0.772520   358.325134   
143549      520.156433   0.969324   378.646454  ...   0.743751   358.882996   
143550      518.341431   0.981317   378.386292  ...   0.709996   358.746521   
143551      518.144714   0.982116   378.901367  ...   0.719511   359.702393   
143552      517.431152   0.972554   379.564880  ...   0.758008   359.798431   

scorer                                                               \
bodyparts                         head_stage                          
coords              y likelihood           x           y likelihood   
0           28.951492   0.000000    9.371573  982.402710   0.000000   
1           28.360371   0.000000    9.244394  982.424316   0.000000   
2           28.429068   0.000000    9.235188  982.434509   0.000000   
3           28.423143   0.000000    9.234289  982.404175   0.000000   
4           28.364849   0.000000    9.320097  982.334961   0.000000   
...               ...        ...         ...         ...        ...   
143548     550.428040   0.571847  414.531860  496.007355   0.848309   
143549     549.892883   0.562917  414.725494  496.876526   0.887016   
143550     548.639221   0.626313  416.623474  495.57

In [46]:
# Save the updated DataFrame back to the HDF5 file, update path and name as wish
h5_file.to_hdf(folder_path + file_name + '_updated.h5', key=file_name, mode='w')